# Explore the Kilter Board SQLite database downloaded via boardlib.

To generate data/tension.db, run the following commands:
- pip install boardlib
- boardlib database tension kilter.db

In [198]:
import sqlite3
from pathlib import Path
import re

In [199]:
DB_PATH = Path("../data/kilter.db")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

## 1. List all tables

In [200]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row[0] for row in cur.fetchall()]
print(f"Tables ({len(tables)}):")
for t in tables:
    count = cur.execute(f"SELECT COUNT(*) FROM [{t}]").fetchone()[0]
    print(f"  {t:40s} {count:>8} rows")

Tables (31):
  android_metadata                                1 rows
  ascents                                         0 rows
  attempts                                       38 rows
  beta_links                                  32139 rows
  bids                                            0 rows
  circuits                                        0 rows
  circuits_climbs                                 0 rows
  climb_cache_fields                         208457 rows
  climb_random_positions                          0 rows
  climb_stats                                348028 rows
  climbs                                     344504 rows
  difficulty_grades                              39 rows
  holes                                        3294 rows
  kits                                          100 rows
  layouts                                         8 rows
  leds                                         7828 rows
  placement_roles                                30 rows
  placements      

## 2. Schema for each table

In [201]:
for t in tables:
    sql = cur.execute(
        "SELECT sql FROM sqlite_master WHERE name=?", (t,)
    ).fetchone()[0]
    print(f"\n{sql};")


CREATE TABLE android_metadata (locale TEXT);

CREATE TABLE ascents (
    uuid TEXT NOT NULL PRIMARY KEY,
    climb_uuid TEXT NOT NULL,
    angle INT UNSIGNED NOT NULL,
    is_mirror BOOLEAN NOT NULL,
    user_id INT UNSIGNED NOT NULL,
    attempt_id INT UNSIGNED NOT NULL,
    bid_count INT UNSIGNED NOT NULL DEFAULT 1,
    quality INT UNSIGNED NOT NULL,
    difficulty INT UNSIGNED NOT NULL,
    is_benchmark INT UNSIGNED NOT NULL DEFAULT 0, -- boolean
    comment TEXT NOT NULL DEFAULT '',
    climbed_at TEXT NOT NULL,
    created_at TEXT NULL DEFAULT NULL,
    FOREIGN KEY (climb_uuid) REFERENCES climbs(uuid) ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (user_id) REFERENCES users(id) ON UPDATE CASCADE ON DELETE CASCADE,
    FOREIGN KEY (attempt_id) REFERENCES attempts(id) ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (difficulty) REFERENCES difficulty_grades(difficulty) ON UPDATE CASCADE ON DELETE RESTRICT
);

CREATE TABLE attempts (
    id INT UNSIGNED NOT NULL PRIMARY 

## 3. Explore board grid positions (holes)

In [202]:
for row in cur.execute("SELECT * FROM holes LIMIT 10"):
    print(dict(row))

# x, y ranges
cur.execute("SELECT product_id, MIN(x), MAX(x), MIN(y), MAX(y), COUNT(*) FROM holes GROUP BY product_id")
for row in cur.fetchall():
    print(f"\nproduct_id={row[0]}: {row[5]} holes, x=[{row[1]}, {row[2]}], y=[{row[3]}, {row[4]}]")

{'id': 1133, 'product_id': 1, 'name': '35,KB1', 'x': 140, 'y': 4, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1134, 'product_id': 1, 'name': '34,KB2', 'x': 136, 'y': 8, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1135, 'product_id': 1, 'name': '33,KB1', 'x': 132, 'y': 4, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1136, 'product_id': 1, 'name': '32,KB2', 'x': 128, 'y': 8, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1137, 'product_id': 1, 'name': '31,KB1', 'x': 124, 'y': 4, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1138, 'product_id': 1, 'name': '30,KB2', 'x': 120, 'y': 8, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1139, 'product_id': 1, 'name': '29,KB1', 'x': 116, 'y': 4, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1140, 'product_id': 1, 'name': '28,KB2', 'x': 112, 'y': 8, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1141, 'product_id': 1, 'name': '27,KB1', 'x': 108, 'y': 4, 'mirrored_hole_id': 0, 'mirror_group': 0}
{'id': 1142, 'product_id': 1

## 4. Hold placements

In [203]:
for row in cur.execute("SELECT * FROM placements LIMIT 10"):
    print(dict(row))

print("\nPlacements per layout:")
for row in cur.execute("""
    SELECT l.name, l.id, COUNT(*) as n_placements
    FROM placements p
    JOIN layouts l ON p.layout_id = l.id
    GROUP BY p.layout_id
"""):
    print(dict(row))

{'id': 1073, 'layout_id': 1, 'hole_id': 1134, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1074, 'layout_id': 1, 'hole_id': 1136, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1075, 'layout_id': 1, 'hole_id': 1138, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1076, 'layout_id': 1, 'hole_id': 1140, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1077, 'layout_id': 1, 'hole_id': 1142, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1078, 'layout_id': 1, 'hole_id': 1144, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1079, 'layout_id': 1, 'hole_id': 1146, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1080, 'layout_id': 1, 'hole_id': 1148, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1081, 'layout_id': 1, 'hole_id': 1150, 'set_id': 1, 'default_placement_role_id': 13}
{'id': 1082, 'layout_id': 1, 'hole_id': 1152, 'set_id': 1, 'default_placement_role_id': 13}

Placements per layout:
{'name': 'Kilter Board Original', 'id': 1, 'n_placements

## 5. Placement roles

In [204]:
for row in cur.execute("SELECT * FROM placement_roles"):
    print(dict(row))

{'id': 12, 'product_id': 1, 'position': 1, 'name': 'start', 'full_name': 'Start', 'led_color': '00FF00', 'screen_color': '00DD00'}
{'id': 13, 'product_id': 1, 'position': 2, 'name': 'middle', 'full_name': 'Middle', 'led_color': '00FFFF', 'screen_color': '00FFFF'}
{'id': 14, 'product_id': 1, 'position': 3, 'name': 'finish', 'full_name': 'Finish', 'led_color': 'FF00FF', 'screen_color': 'FF00FF'}
{'id': 15, 'product_id': 1, 'position': 4, 'name': 'foot', 'full_name': 'Foot Only', 'led_color': 'FFA500', 'screen_color': 'FFA500'}
{'id': 20, 'product_id': 2, 'position': 1, 'name': 'start', 'full_name': 'Start', 'led_color': '00FF00', 'screen_color': '00DD00'}
{'id': 21, 'product_id': 2, 'position': 2, 'name': 'middle', 'full_name': 'Middle', 'led_color': '00FFFF', 'screen_color': '00FFFF'}
{'id': 22, 'product_id': 2, 'position': 3, 'name': 'finish', 'full_name': 'Finish', 'led_color': 'FF00FF', 'screen_color': 'FF00FF'}
{'id': 23, 'product_id': 2, 'position': 4, 'name': 'foot', 'full_name': 

## 6. LEDs

In [205]:
for row in cur.execute("SELECT * FROM leds LIMIT 10"):
    print(dict(row))

{'id': 2568, 'product_size_id': 7, 'hole_id': 1133, 'position': 0}
{'id': 2569, 'product_size_id': 7, 'hole_id': 1134, 'position': 1}
{'id': 2570, 'product_size_id': 7, 'hole_id': 1135, 'position': 2}
{'id': 2571, 'product_size_id': 7, 'hole_id': 1136, 'position': 3}
{'id': 2572, 'product_size_id': 7, 'hole_id': 1137, 'position': 4}
{'id': 2573, 'product_size_id': 7, 'hole_id': 1138, 'position': 5}
{'id': 2574, 'product_size_id': 7, 'hole_id': 1139, 'position': 6}
{'id': 2575, 'product_size_id': 7, 'hole_id': 1140, 'position': 7}
{'id': 2576, 'product_size_id': 7, 'hole_id': 1141, 'position': 8}
{'id': 2577, 'product_size_id': 7, 'hole_id': 1142, 'position': 9}


## 7. Climbs

In [206]:
for row in cur.execute("SELECT * FROM climbs LIMIT 5"):
    d = dict(row)
    if 'frames' in d and d['frames'] and len(str(d['frames'])) > 80:
        d['frames'] = str(d['frames']) + '...'
    print(d)

{'uuid': '002047402B6941CEA5ED7BB09FBFE14D', 'layout_id': 1, 'setter_id': 1051, 'setter_username': 'kilterjackie', 'name': '4/26 Harder Than It Should Be', 'description': '', 'hsm': 3, 'edge_left': 40, 'edge_right': 136, 'edge_bottom': 4, 'edge_top': 152, 'angle': None, 'frames_count': 1, 'frames_pace': 0, 'frames': 'p1145r12p1146r12p1149r13p1186r13p1201r13p1256r15p1267r13p1276r15p1307r13p1321r13p1322r13p1343r13p1356r13p1392r14p1454r15p1455r15p1456r15p1457r15p1506r15p1523r15p1527r15p1533r15p1535r15...', 'is_draft': 0, 'is_listed': 0, 'created_at': '2018-04-27 03:31:54.979371', 'is_nomatch': 0}
{'uuid': '002ED50792A94E5EB2127D59E167B2EE', 'layout_id': 1, 'setter_id': 17760, 'setter_username': 'bctyner', 'name': 'what kind of triangle', 'description': 'Make sure you angle your body just right', 'hsm': 3, 'edge_left': 4, 'edge_right': 140, 'edge_bottom': 4, 'edge_top': 152, 'angle': None, 'frames_count': 1, 'frames_pace': 0, 'frames': 'p1123r12p1139r13p1155r12p1171r13p1187r13p1203r13p1219

## 8. Products and Product sizes

In [207]:
for row in cur.execute("SELECT * FROM products"):
    print(dict(row))

for row in cur.execute("SELECT * FROM product_sizes"):
    print(dict(row))

{'id': 1, 'name': 'Kilter Board Original', 'is_listed': 1, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 35}
{'id': 7, 'name': 'Kilter Board Homewall', 'is_listed': 1, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 35}
{'id': 2, 'name': 'JUUL', 'is_listed': 1, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 35}
{'id': 3, 'name': 'Demo Board', 'is_listed': 0, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 35}
{'id': 4, 'name': 'BKB Board', 'is_listed': 0, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 35}
{'id': 6, 'name': 'Tycho', 'is_listed': 1, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 120}
{'id': 5, 'name': 'Spire', 'is_listed': 0, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 35}
{'id': 8, 'product_id': 1, 'edge_left': 24, 'edge_right': 120, 'edge_bottom': 0, 'edge_top': 156, 'name': '8 x 12', 'description': 'Home', 'image_filename': 'product_sizes/8-t

## 9. Layouts

In [208]:
for row in cur.execute("SELECT * FROM layouts"):
    print(dict(row))

cur.execute("""
    SELECT l.name, l.id, COUNT(*) as n_placements
    FROM placements p
    JOIN layouts l ON p.layout_id = l.id
    GROUP BY p.layout_id
""")
for row in cur.fetchall():
    print(dict(row))

{'id': 1, 'product_id': 1, 'name': 'Kilter Board Original', 'instagram_caption': '', 'is_mirrored': 0, 'is_listed': 1, 'password': None, 'created_at': '2015-12-31 20:02:31.000000'}
{'id': 2, 'product_id': 2, 'name': 'JUUL', 'instagram_caption': '', 'is_mirrored': 0, 'is_listed': 1, 'password': 'freedom', 'created_at': '2019-05-26 17:35:44.426770'}
{'id': 3, 'product_id': 3, 'name': 'Standard Medium', 'instagram_caption': "Kilter's Demo Board.", 'is_mirrored': 0, 'is_listed': 0, 'password': 'sales', 'created_at': '2019-07-29 23:35:36.435295'}
{'id': 4, 'product_id': 4, 'name': 'BKBBoard Level 1', 'instagram_caption': '', 'is_mirrored': 0, 'is_listed': 0, 'password': 'BKBlove', 'created_at': '2019-10-21 20:43:47.467242'}
{'id': 6, 'product_id': 6, 'name': 'Tycho Complete', 'instagram_caption': '', 'is_mirrored': 0, 'is_listed': 0, 'password': 'explore', 'created_at': '2020-07-21 20:48:59.728468'}
{'id': 7, 'product_id': 6, 'name': 'Tycho 2020', 'instagram_caption': '', 'is_mirrored': 0, 

## 10. Parse a sample climb's frames string

Format: p<placement_id>r<role_id>p<placement_id>r<role_id>...

In [209]:
sample = cur.execute("""
    SELECT c.uuid, c.name, c.frames, cs.display_difficulty, dg.boulder_name
    FROM climbs c
    JOIN climb_stats cs ON c.uuid = cs.climb_uuid AND cs.angle = 30
    JOIN difficulty_grades dg ON CAST(cs.display_difficulty AS INT) = dg.difficulty
    WHERE c.layout_id = 1
    AND c.frames IS NOT NULL AND c.frames != ''
    AND c.is_listed = 1
    LIMIT 1
""").fetchone()
if sample:
    climb_uuid, name, frames, difficulty, grade = sample
    print(f"Climb: {name} (uuid={climb_uuid})")
    print(f"Grade at 30°: {grade} (difficulty={difficulty})")
    print(f"Raw frames: {frames}")

    pairs = re.findall(r'p(\d+)r(\d+)', frames)
    print(f"\nParsed {len(pairs)} placements:")
    for placement_id, role_id in pairs:
        row = cur.execute("""
            SELECT p.id as placement_id, p.hole_id, h.x, h.y, h.name as hole_name,
                   pr.full_name as role_name
            FROM placements p
            JOIN holes h ON p.hole_id = h.id
            LEFT JOIN placement_roles pr ON pr.id = ?
            WHERE p.id = ?
        """, (role_id, placement_id)).fetchone()
        if row:
            print(f"  placement={row['placement_id']}, hole={row['hole_name']}, "
                  f"pos=({row['x']}, {row['y']}), role={row['role_name']}")

Climb: trash pile (uuid=004283D308F14EA7B3DD5C2FE584809D)
Grade at 30°: 6c/V5 (difficulty=20.0)
Raw frames: p1145r12p1216r13p1233r13p1271r13p1339r13p1357r13p1387r14

Parsed 7 placements:
  placement=1145, hole=10,7, pos=(40, 40), role=Start
  placement=1216, hole=16,15, pos=(64, 72), role=Middle
  placement=1233, hole=16,17, pos=(64, 80), role=Middle
  placement=1271, hole=24,21, pos=(96, 96), role=Middle
  placement=1339, hole=24,29, pos=(96, 128), role=Middle
  placement=1357, hole=26,31, pos=(104, 136), role=Middle
  placement=1387, hole=18,35, pos=(72, 152), role=Finish


## 11. Number of climbs per layout

In [210]:
for row in cur.execute("""
    SELECT l.name, l.id, COUNT(*) as n_climbs
    FROM climbs c JOIN layouts l ON c.layout_id = l.id
    GROUP BY c.layout_id
"""):
    print(dict(row))

{'name': 'Kilter Board Original', 'id': 1, 'n_climbs': 315357}
{'name': 'JUUL', 'id': 2, 'n_climbs': 160}
{'name': 'Standard Medium', 'id': 3, 'n_climbs': 2}
{'name': 'Spire', 'id': 5, 'n_climbs': 637}
{'name': 'Tycho Complete', 'id': 6, 'n_climbs': 6}
{'name': 'Tycho 2020', 'id': 7, 'n_climbs': 12}
{'name': 'Kilter Board Homewall', 'id': 8, 'n_climbs': 28330}


## 12. Kilter coordinate system

In [211]:
cur.execute("""
    SELECT MIN(x), MAX(x), MIN(y), MAX(y), COUNT(*)
    FROM holes WHERE product_id = 5
""")
xmin, xmax, ymin, ymax, n = cur.fetchone()
print(f"Kilter holes: {n} total, x=[{xmin}, {xmax}], y=[{ymin}, {ymax}]")

Kilter holes: 544 total, x=[4, 124], y=[-12, 276]


## 13. Sets

In [212]:
for row in cur.execute("SELECT * FROM sets"):
    print(dict(row))

{'id': 1, 'name': 'Bolt Ons', 'hsm': 1}
{'id': 20, 'name': 'Screw Ons', 'hsm': 2}
{'id': 21, 'name': 'JUUL All', 'hsm': 1}
{'id': 22, 'name': 'Demo Holds', 'hsm': 1}
{'id': 23, 'name': 'BKB All', 'hsm': 1}
{'id': 24, 'name': 'Spire All', 'hsm': 1}
{'id': 25, 'name': 'Orbit', 'hsm': 1}
{'id': 26, 'name': 'Mainline', 'hsm': 1}
{'id': 27, 'name': 'Auxiliary', 'hsm': 2}
{'id': 28, 'name': 'Mainline Kickboard', 'hsm': 4}
{'id': 29, 'name': 'Auxiliary Kickboard', 'hsm': 8}


## 14. Difficulty grades

In [213]:
for row in cur.execute("SELECT * FROM difficulty_grades"):
    print(dict(row))

{'difficulty': 1, 'boulder_name': '1a/V0', 'route_name': '2b/5.1', 'is_listed': 0}
{'difficulty': 2, 'boulder_name': '1b/V0', 'route_name': '2c/5.2', 'is_listed': 0}
{'difficulty': 3, 'boulder_name': '1c/V0', 'route_name': '3a/5.3', 'is_listed': 0}
{'difficulty': 4, 'boulder_name': '2a/V0', 'route_name': '3b/5.3', 'is_listed': 0}
{'difficulty': 5, 'boulder_name': '2b/V0', 'route_name': '3c/5.4', 'is_listed': 0}
{'difficulty': 6, 'boulder_name': '2c/V0', 'route_name': '4a/5.5', 'is_listed': 0}
{'difficulty': 7, 'boulder_name': '3a/V0', 'route_name': '4b/5.6', 'is_listed': 0}
{'difficulty': 8, 'boulder_name': '3b/V0', 'route_name': '4c/5.7', 'is_listed': 0}
{'difficulty': 9, 'boulder_name': '3c/V0', 'route_name': '5a/5.8', 'is_listed': 0}
{'difficulty': 10, 'boulder_name': '4a/V0', 'route_name': '5b/5.9', 'is_listed': 1}
{'difficulty': 11, 'boulder_name': '4b/V0', 'route_name': '5c/5.10a', 'is_listed': 1}
{'difficulty': 12, 'boulder_name': '4c/V0', 'route_name': '6a/5.10b', 'is_listed': 

## 15. 30 vs 40 degrees

In [214]:
cur.execute("""
    SELECT dg.boulder_name, COUNT(*) as n
    FROM climb_stats cs
    JOIN climbs c ON c.uuid = cs.climb_uuid
    JOIN difficulty_grades dg ON CAST(cs.display_difficulty AS INT) = dg.difficulty
    WHERE cs.angle = 30 AND c.layout_id = 1
    GROUP BY CAST(cs.display_difficulty AS INT)
    ORDER BY CAST(cs.display_difficulty AS INT)
""")
print("Climbs at 30° by grade:")
for row in cur.fetchall():
    print(f"  {row[0]:20s} {row[1]:>6}")
    
cur.execute("""
    SELECT dg.boulder_name, COUNT(*) as n
    FROM climb_stats cs
    JOIN climbs c ON c.uuid = cs.climb_uuid
    JOIN difficulty_grades dg ON CAST(cs.display_difficulty AS INT) = dg.difficulty
    WHERE cs.angle = 40 AND c.layout_id = 1
    GROUP BY CAST(cs.display_difficulty AS INT)
    ORDER BY CAST(cs.display_difficulty AS INT)
""")
print("Climbs at 40° by grade:")
for row in cur.fetchall():
    print(f"  {row[0]:20s} {row[1]:>6}")

Climbs at 30° by grade:
  4a/V0                   992
  4b/V0                   752
  4c/V0                   681
  5a/V1                  1548
  5b/V1                  1703
  5c/V2                  3374
  6a/V3                  4789
  6a+/V3                 4003
  6b/V4                  5417
  6b+/V4                 4214
  6c/V5                  4692
  6c+/V5                 3321
  7a/V6                  3419
  7a+/V7                 1861
  7b/V8                   906
  7b+/V8                  422
  7c/V9                   184
  7c+/V10                  61
  8a/V11                   18
  8a+/V12                   9
  8b/V13                    6
  8b+/V14                   6
  8c/V15                    4
  8c+/V16                  37
Climbs at 40° by grade:
  4a/V0                  1058
  4b/V0                   795
  4c/V0                   696
  5a/V1                  1484
  5b/V1                  1709
  5c/V2                  2881
  6a/V3                  5095
  6a+/V3              

In [215]:
conn.close()